# Model app integration test

This notebook calls `run_model_app()` in `gradientdescentfarmers/app.py` with the editable `USER_PARAMETERS` dictionary. The final output is **one DataFrame and one CSV**: each row is one phase of one rotation option. `rotation_id` identifies the alternative, while `rotation_step` and `rotation_phase_count` show its ordered phases. Every row retains the terminal WOFOST variables, SIAP calibration, harvest-price estimate, gross value, and the entire daily WOFOST trajectory as JSON in `wofost_daily_json`.

The example offers a three-phase maize → faba bean → maize option and a two-phase faba bean → maize option. It reads the farm profile and SIAP from GCS, but uses **invented historical and future weather, soil, dates, and areas** for a wiring test. Its yields and values are not validated field forecasts. Supply matching historical weather, measured soil, a documented future-weather scenario, actual crop dates, and cultivated areas for a real scenario. Set Google Application Default Credentials before starting the notebook kernel.

`rotation_options` is an input schedule; this pipeline does not choose crops or carry soil state across its separate season runs. `water_mode` may be `potential`, `rainfed`, or `irrigated`. For irrigation, set a system name and add `irrigation_events` to every historical and planned phase. For potential production, set soil and irrigation settings to `None`.


In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd

repo_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                  if (path / "gradientdescentfarmers" / "model_builder").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Open this notebook within the repository or install the package")
sys.path.insert(0, str(repo_root))

from gradientdescentfarmers.app import PipelineConfig, run_model_app
config_path = repo_root / "resources" / "application.conf"

In [ ]:
# Invented weather is only a wiring check. The 2022–2023 portion lets the
# faba-bean simulation overlap the available SIAP calibration years.
def synthetic_weather(start, end):
    return pd.DataFrame({
        "date": pd.date_range(start, end), "tmin_c": 12.0, "tmax_c": 27.0,
        "precip_mm": 2.0, "rh_pct": 65.0, "wind_m_s": 2.0,
        "solar_mj_m2_day": 19.0,
    })

USER_PARAMETERS = {
    "selected_products": ["maize", "fababean"],
    "water_mode": "rainfed",
    "irrigation_system": None,
    "historical_weather": synthetic_weather("2022-01-01", "2025-12-31"),
    "historical_seasons": [
        {"crop": "fababean", "sowing_date": "2022-03-01", "harvest_date": "2022-09-01"},
        {"crop": "fababean", "sowing_date": "2023-03-01", "harvest_date": "2023-09-01"},
        {"crop": "maize", "sowing_date": "2024-05-15", "harvest_date": "2024-12-15"},
        {"crop": "maize", "sowing_date": "2025-05-15", "harvest_date": "2025-12-15"},
    ],
    "rotation_options": [
        {"rotation_id": "maize_faba_maize", "phases": [
            {"crop": "maize", "sowing_date": "2027-05-15",
             "harvest_date": "2027-12-15", "area_ha": 1.0},
            {"crop": "fababean", "sowing_date": "2028-01-15",
             "harvest_date": "2028-06-15", "area_ha": 1.0},
            {"crop": "maize", "sowing_date": "2029-05-15",
             "harvest_date": "2029-12-15", "area_ha": 1.0},
        ]},
        {"rotation_id": "faba_maize", "phases": [
            {"crop": "fababean", "sowing_date": "2027-01-15",
             "harvest_date": "2027-06-15", "area_ha": 1.0},
            {"crop": "maize", "sowing_date": "2028-05-15",
             "harvest_date": "2028-12-15", "area_ha": 1.0},
        ]},
    ],
    "soil_parameters": {
        "SMW": 0.1, "SMFCF": 0.3, "SM0": 0.4, "RDMSOL": 100.0,
        "CRAIRC": 0.06, "SOPE": 10.0, "KSUB": 10.0, "K0": 10.0,
    },
    "initial_available_water_cm": 10.0,
    "future_weather": synthetic_weather("2027-01-01", "2029-12-31"),
    "future_weather_source": "SYNTHETIC_SMOKE",
}

# For irrigated mode, add an event inside every season/phase dictionary, e.g.:
# {"date": "2027-06-01", "amount_mm": 40.0, "efficiency": 0.8}


In [ ]:
file_config = PipelineConfig.from_file(config_path)
resolved = {**file_config.model, **USER_PARAMETERS}
print("Products:", resolved["selected_products"])
print("Water mode:", resolved["water_mode"])
for option in resolved["rotation_options"]:
    print("Rotation option:", option["rotation_id"], "phases:", len(option["phases"]))
print("Historical seasons:", len(resolved["historical_seasons"]))
print("Future weather source:", resolved["future_weather_source"])
print("SMOKE TEST: historical/future weather, soil, dates, and areas are invented.")


In [ ]:
result = run_model_app(USER_PARAMETERS, config_path=config_path)
final_df = result.final_df
assert len(final_df) == sum(len(option["phases"]) for option in USER_PARAMETERS["rotation_options"])
assert final_df.groupby("rotation_id").rotation_step.apply(list).to_dict() == {
    "maize_faba_maize": [1, 2, 3], "faba_maize": [1, 2],
}
assert all(json.loads(value) for value in final_df.wofost_daily_json)
preview = [
    "rotation_id", "rotation_step", "rotation_phase_count", "crop",
    "sowing_date", "harvest_date", "SM", "summary_NuptakeTotal",
    "terminal_RAINT", "calibrated_t_ha", "estimated_tonnes",
    "estimated_price_mxn_t", "estimated_gross_value_mxn",
]
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(final_df[preview])
print("Full final DataFrame columns:", len(final_df.columns))
print("Daily WOFOST records by phase:", [len(json.loads(value)) for value in final_df.wofost_daily_json])
print("Months through last planned harvest:", result.horizon_months)


In [ ]:
SAVE_OUTPUT = True
if SAVE_OUTPUT:
    output_dir = repo_root / "data" / "model_builder"
    name = ("app_synthetic_smoke_final.csv" if resolved["future_weather_source"] == "SYNTHETIC_SMOKE"
            else "app_model_result_final.csv")
    output_path = result.save_final_csv(output_dir / name)
    print("Saved one consolidated CSV:", output_path)
